## GeoPoll Questionnaire Validator
Validates a country questionnaire against a reference (template or previous round).
Built with Polars for speed. Supports GeoPoll format (Kobo to be added later).##

In [1]:
import re
import yaml
import polars as pl
import openpyxl
from pathlib import Path
from dataclasses import dataclass
from typing import Literal, Optional


## Step 1 — Configuration
Set your file paths and validation mode here. This is the only cell you need to edit between runs.
`reference_mode` options:
- `"latest_template"` — compare against the most recent official template in your repo
- `"custom_reference"` — compare against any file you specify
- `"previous_round"` — compare against the previous round questionnaire

In [2]:
ReferenceMode = Literal["latest_template", "custom_reference", "previous_round"]
Enumerator    = Literal["geopoll", "kobo"]
Language      = Literal["EN", "FR", "ES", "AR", "PT"]

@dataclass
class ValidationConfig:
    template_repo          : Path
    working_dir            : Path
    questionnaire_file     : str
    reference_mode         : ReferenceMode
    enumerator             : Enumerator
    language               : Language
    custom_reference_file  : Optional[str]  = None
    previous_round_file    : Optional[str]  = None
    output_subfolder       : str            = "output"
    treat_blank_as_no      : bool           = False   # treat blank Mandatory cell as "no"
    critical_sets_file     : Optional[Path] = None    # path to critical_sets.yaml; None = auto-detect

cfg = ValidationConfig(
    template_repo        = Path(r"C:\Users\edoar\Food and Agriculture Organization\OER - 01. DIEM Monitoring\03. Toolbox\02. Questionnaires"),
    working_dir          = Path(r"C:\Questionnaire_Validation"),
    questionnaire_file   = "GeoPoll_FAO_FR_DRC_R11_test.xlsx",
    reference_mode       = "latest_template",
    enumerator           = "geopoll",
    language             = "EN",
    custom_reference_file = None,
    previous_round_file   = None,
)


## Step 2 — Reference file resolution
Finds the correct template file automatically based on enumerator and language,
or uses the file you specified in the config above.

In [3]:
# Maps language code (cfg.language) to the column name in the questionnaire Excel file
LANGUAGE_COLUMN: dict[str, str] = {
    "EN": "English",
    "FR": "French",
    "ES": "Spanish",
    "AR": "Arabic",
    "PT": "Portuguese",
}


def extract_date_from_name(path: Path) -> int:
    """
    Pulls the 8-digit date out of a filename like:
      household_questionnaire_geopoll_EN_template_20250708_ISO3.xlsx
                                                    ^^^^^^^^
    Returns 0 if no date is found, so the file sorts last.
    """
    m = re.search(r"template_(\d{8})", path.name)
    return int(m.group(1)) if m else 0


def find_latest_template(template_repo: Path, enumerator: str, language: str) -> Path:
    """
    Scans the template repo folder and returns the most recent template
    matching the given enumerator (geopoll/kobo) and language (EN/FR/etc.).
    Sorts by date in filename first, then by file modification time as fallback.
    """
    matches = [
        p for p in template_repo.glob("*.xlsx")
        if enumerator.lower() in p.name.lower()
        and f"_{language.lower()}_" in p.name.lower()
        and "template" in p.name.lower()
    ]
    if not matches:
        raise FileNotFoundError(
            f"No template found for enumerator='{enumerator}', language='{language}' in:\n  {template_repo}"
        )
    matches.sort(key=lambda p: (extract_date_from_name(p), p.stat().st_mtime), reverse=True)
    return matches[0]


def resolve_reference_file(cfg: ValidationConfig) -> Path:
    """Returns the path to the reference file based on reference_mode in the config."""
    if cfg.reference_mode == "latest_template":
        return find_latest_template(cfg.template_repo, cfg.enumerator, cfg.language)
    if cfg.reference_mode == "custom_reference":
        if not cfg.custom_reference_file:
            raise ValueError("custom_reference_file must be set when reference_mode='custom_reference'")
        return cfg.working_dir / cfg.custom_reference_file
    if cfg.reference_mode == "previous_round":
        if not cfg.previous_round_file:
            raise ValueError("previous_round_file must be set when reference_mode='previous_round'")
        return cfg.working_dir / cfg.previous_round_file
    raise ValueError(f"Unknown reference_mode: '{cfg.reference_mode}'")


def load_critical_sets(cfg: ValidationConfig) -> dict:
    """
    Loads validation rules from critical_sets.yaml.
    Search order:
      1. cfg.critical_sets_file (if set)
      2. critical_sets.yaml in the current working directory
      3. critical_sets.yaml in a scripts/ subfolder of the current directory

    Returns a dict with keys: exact_sets, min_count_sets, crop_harvest.
    If the file is not found, structural validation is disabled (empty rules returned).
    """
    candidates = [
        Path(cfg.critical_sets_file) if cfg.critical_sets_file else None,
        Path.cwd() / "critical_sets.yaml",
        Path.cwd() / "scripts" / "critical_sets.yaml",
    ]
    for path in candidates:
        if path is not None and path.exists():
            with open(path, "r", encoding="utf-8") as f:
                data = yaml.safe_load(f)
            print(f"Rules loaded  : {path}")
            return {
                "exact_sets"    : data.get("exact_sets", {}),
                "min_count_sets": data.get("min_count_sets", {}),
                "crop_harvest"  : data.get("crop_harvest", {}),
            }
    print("⚠️  critical_sets.yaml not found — structural validation disabled.")
    print(f"   Searched: {[str(p) for p in candidates if p is not None]}")
    return {"exact_sets": {}, "min_count_sets": {}, "crop_harvest": {}}


def prepare_run(cfg: ValidationConfig) -> dict:
    """
    Creates output directories and resolves all file paths.
    Returns a dict with the three paths the rest of the notebook needs.
    Prints a summary so you can confirm the right files were picked up.
    """
    cfg.working_dir.mkdir(parents=True, exist_ok=True)
    output_dir = cfg.working_dir / cfg.output_subfolder
    output_dir.mkdir(parents=True, exist_ok=True)

    questionnaire_path = cfg.working_dir / cfg.questionnaire_file
    reference_path     = resolve_reference_file(cfg)

    if not questionnaire_path.exists():
        raise FileNotFoundError(f"Questionnaire file not found:\n  {questionnaire_path}")
    if not reference_path.exists():
        raise FileNotFoundError(f"Reference file not found:\n  {reference_path}")

    result_file = output_dir / f"{questionnaire_path.stem.split()[0]}_{cfg.enumerator}_validated.xlsx"

    run = {
        "questionnaire_path" : questionnaire_path,
        "reference_path"     : reference_path,
        "result_file"        : result_file,
    }

    print("Questionnaire :", questionnaire_path)
    print("Reference     :", reference_path)
    print("Output        :", result_file)
    return run


run   = prepare_run(cfg)
rules = load_critical_sets(cfg)

# Language-specific column for option label comparison
text_col = LANGUAGE_COLUMN.get(cfg.language, "English")
print(f"Language      : {cfg.language} → comparing '{text_col}' option labels")


Questionnaire : C:\Questionnaire_Validation\GeoPoll_FAO_FR_DRC_R11_test.xlsx
Reference     : C:\Users\edoar\Food and Agriculture Organization\OER - 01. DIEM Monitoring\03. Toolbox\02. Questionnaires\household_questionnaire_geopoll_EN_template_20250708_ISO3.xlsx
Output        : C:\Questionnaire_Validation\output\GeoPoll_FAO_FR_DRC_R11_test_geopoll_validated.xlsx
Rules loaded  : c:\Users\edoar\WORK\FAO\repo\questionnaire_validation_revision\scripts\critical_sets.yaml
Language      : EN → comparing 'English' option labels


## Step 3 — Reading and normalising the survey sheet
`read_survey_sheet` loads the Excel file into a Polars DataFrame and tracks the
original Excel row number so every mismatch can be traced back to an exact line.

`build_questions_df` then selects only the columns we care about for validation
and casts everything to clean strings (strip whitespace, fill nulls with `""`).

In [4]:
CORE_COLUMNS = [
    "Q Name",
    "Suggested Qname",
    "English",
    "French",
    "Spanish",
    "Arabic",
    "Portuguese",
    "Q Type",
    "Mandatory",
    "Default skip patterns & conditional ",   # note: trailing space is in the real header
    "Specify skip pattern variable (from blue text)",
    "Additional notes",
    "excel_row",
    "source_file",
]


def read_survey_sheet(path: Path, sheet_name: str = "survey", header_row: int = 3) -> pl.DataFrame:
    """
    Reads the survey sheet from an xlsx file into a Polars DataFrame.

    header_row=3 because the GeoPoll template has two title rows above the
    real column headers.

    Two columns are added automatically:
      excel_row   — the actual row number in the Excel file (for tracing mismatches)
      source_file — the filename (tells you which file a row came from)
    """
    wb = openpyxl.load_workbook(path, data_only=True, read_only=True)

    ws = next(
        (wb[n] for n in wb.sheetnames if n.strip().lower() == sheet_name.lower()),
        None
    )
    if ws is None:
        raise KeyError(f"Sheet '{sheet_name}' not found. Available sheets: {wb.sheetnames}")

    row_iter = ws.iter_rows(values_only=True)
    for _ in range(header_row - 1):
        next(row_iter)   # skip the two title rows

    raw_headers = next(row_iter)
    headers = [
        str(h).replace("\n", " ").strip() if h is not None else f"unnamed_{i}"
        for i, h in enumerate(raw_headers, 1)
    ]

    rows      = []
    excel_row = header_row + 1

    for values in row_iter:
        if all(v is None for v in values):
            excel_row += 1
            continue
        row_dict = {headers[i]: values[i] for i in range(len(headers))}
        row_dict["excel_row"]   = excel_row
        row_dict["source_file"] = Path(path).name
        rows.append(row_dict)
        excel_row += 1

    df = pl.DataFrame(rows)
    df = df.filter(pl.col("Q Name").is_not_null())
    return df.with_columns(pl.col("Q Name").cast(pl.Utf8).str.strip_chars())


def build_questions_df(df: pl.DataFrame) -> pl.DataFrame:
    """
    Selects and cleans the core validation columns.
    Warns (does not crash) if an expected column is missing — silent drops
    would make mismatches invisible.
    """
    missing = [
        c for c in CORE_COLUMNS
        if c not in df.columns and c not in ("excel_row", "source_file")
    ]
    if missing:
        print(f"⚠️  Expected columns not found (will be skipped): {missing}")

    available = [c for c in CORE_COLUMNS if c in df.columns]
    out = df.select(available)

    for c in out.columns:
        if c != "excel_row":
            out = out.with_columns(
                pl.col(c).cast(pl.Utf8).fill_null("").str.strip_chars()
            )
    return out

## Step 4 — Exploding answer options
Instead of comparing the full English text block as one string (old approach),
we parse each question's numbered options into individual rows.

This means the diff can say **"option 3 of `resp_gender` changed from X to Y"**
rather than just **"`resp_gender` English text differs"** — giving you the exact
location of the change.

The two questions with no parseable options (`crp_main`, `crp_salesmain`) are
expected: their options are injected dynamically at run time. They are handled
by the presence/absence checks, not here.

In [5]:
OPTION_BEARING_TYPES = {
    "Single Choice",
    "Open Ended-Single Choice",
    "Open Ended - Single Choice",
    "Select All That Apply",
    "Open Ended-Select All That Apply",
    "Open Ended - Select All That Apply",
    "Open Ended - Select All That Apply ",   # trailing-space variant seen in real files
}

# Matches blocks like:
#   1) Label text
#   2) Next label
#
# (?m)    — ^ matches the start of each line
# DOTALL  — . also matches newlines, so multi-line labels are captured in full
OPTION_PATTERN = re.compile(r"(?m)^\s*(\d+)\)\s*(.*?)(?=^\s*\d+\)|\Z)", re.DOTALL)


def parse_options(text: str) -> list[tuple[int, str]]:
    """
    Parses numbered options from a question text cell.
    Returns a list of (option_code, option_label) tuples.

    Labels are whitespace-collapsed but otherwise kept raw —
    

    Example:
        input:  "What is your gender?\\n\\n1)Male\\n2)Female\\n3)DON'T KNOW"
        output: [(1, "Male"), (2, "Female"), (3, "DON'T KNOW")]
    """
    if not text or not isinstance(text, str):
        return []
    results = []
    for m in OPTION_PATTERN.finditer(text):
        code  = int(m.group(1))
        label = " ".join(m.group(2).split())   # collapse newlines/extra spaces
        if label:
            results.append((code, label))
    return results


def explode_options(questions_df: pl.DataFrame, text_col: str = "English") -> pl.DataFrame:
    """
    Turns the questions DataFrame (one row per question) into an options
    DataFrame (one row per answer option).

    Output columns:
        Q Name, Q Type, option_code (Int64), option_label (Utf8),
        excel_row, source_file
    """
    EMPTY_SCHEMA = {
        "Q Name"      : pl.Utf8,
        "Q Type"      : pl.Utf8,
        "option_code" : pl.Int64,
        "option_label": pl.Utf8,
        "excel_row"   : pl.Int64,
        "source_file" : pl.Utf8,
    }

    if text_col not in questions_df.columns:
        print(f"⚠️  Column '{text_col}' not found — returning empty options frame.")
        return pl.DataFrame(schema=EMPTY_SCHEMA)

    df = (
        questions_df.filter(pl.col("Q Type").is_in(list(OPTION_BEARING_TYPES)))
        if "Q Type" in questions_df.columns
        else questions_df
    )

    carry_cols = [c for c in ["Q Name", "Q Type", "excel_row", "source_file"] if c in df.columns]
    rows = []

    for row in df.select(carry_cols + [text_col]).iter_rows(named=True):
        text = row.get(text_col) or ""
        for code, label in parse_options(text):
            out_row = {c: row.get(c) for c in carry_cols}
            out_row["option_code"]  = code
            out_row["option_label"] = label
            rows.append(out_row)

    if not rows:
        return pl.DataFrame(schema=EMPTY_SCHEMA)

    return pl.DataFrame(rows).with_columns(
        pl.col("option_code").cast(pl.Int64),
        pl.col("option_label").cast(pl.Utf8),
    )

In [6]:
current_raw   = read_survey_sheet(run["questionnaire_path"])
reference_raw  = read_survey_sheet(run["reference_path"])

current_questions   = build_questions_df(current_raw)
reference_questions  = build_questions_df(reference_raw)

current_options   = explode_options(current_questions,  text_col=text_col)
reference_options  = explode_options(reference_questions, text_col=text_col)

print("--- current questionnaire ---")
print(f"  questions : {current_questions.shape}")
print(f"  options   : {current_options.shape}")

print("\n--- reference file ---")
print(f"  questions : {reference_questions.shape}")
print(f"  options   : {reference_options.shape}")

print("\n--- sample: resp_gender options (current) ---")
print(current_options.filter(pl.col("Q Name") == "resp_gender"))


⚠️  Expected columns not found (will be skipped): ['French', 'Spanish', 'Arabic', 'Portuguese', 'Default skip patterns & conditional ', 'Specify skip pattern variable (from blue text)', 'Additional notes']
⚠️  Expected columns not found (will be skipped): ['French', 'Spanish', 'Arabic', 'Portuguese', 'Default skip patterns & conditional ', 'Additional notes']
--- current questionnaire ---
  questions : (196, 7)
  options   : (1090, 6)

--- reference file ---
  questions : (180, 8)
  options   : (845, 6)

--- sample: resp_gender options (current) ---
shape: (4, 6)
┌─────────────┬───────────────┬───────────┬───────────────────────────┬─────────────┬──────────────┐
│ Q Name      ┆ Q Type        ┆ excel_row ┆ source_file               ┆ option_code ┆ option_label │
│ ---         ┆ ---           ┆ ---       ┆ ---                       ┆ ---         ┆ ---          │
│ str         ┆ str           ┆ i64       ┆ str                       ┆ i64         ┆ str          │
╞═════════════╪═══════════

## Step 6 — Normalisation helpers
Two helpers used by the comparison functions below:

- `normalize_text_expr` — lowercases, removes punctuation, collapses spaces.
  Used for option label comparison so "DON'T KNOW" and "Don't know" are treated as equal.
- `normalize_mandatory_expr` — maps any variant of yes/no/true/false/1/0 to a
  canonical `"yes"` or `"no"` so formatting differences don't create false mismatches.

In [7]:
def normalize_text_expr(col_name: str) -> pl.Expr:
    """
    Polars expression that lowercases a string column, removes punctuation,
    and collapses whitespace. Apply as an alias so the original column is preserved.

    Usage:
        df.with_columns(normalize_text_expr("option_label").alias("option_label_norm"))
    """
    return (
        pl.col(col_name)
        .cast(pl.Utf8)
        .fill_null("")
        .str.to_lowercase()
        .str.replace_all(r"[^\w\s]", "")   # remove punctuation
        .str.replace_all(r"\s+", " ")       # collapse multiple spaces
        .str.strip_chars()
    )


def normalize_mandatory_expr(col_name: str) -> pl.Expr:
    """
    Polars expression that maps any common yes/no variant to canonical "yes"/"no"/""
    so that "Yes", "YES", "y", "true", "1" are all treated as equivalent.

    Using when/then/otherwise keeps this inside the Polars execution engine
    (faster and parallelisable) vs a Python lambda with map_elements.
    """
    cleaned = (
        pl.col(col_name)
        .cast(pl.Utf8)
        .fill_null("")
        .str.to_lowercase()
        .str.strip_chars()
    )
    return (
        pl.when(cleaned.is_in(["yes", "y", "true", "1"])).then(pl.lit("yes"))
        .when(cleaned.is_in(["no",  "n", "false", "0"])).then(pl.lit("no"))
        .otherwise(pl.lit(""))
    )

## Step 7 — Comparison and validation functions
Seven focused functions, each returning a tidy DataFrame in the common issues schema:

| Function | What it checks | Source of rules |
|---|---|---|
| `compare_question_presence` | Questions added or removed vs reference; `o_*` optional questions tagged as *info* | reference file |
| `compare_mandatory` | Mandatory flag changed for any shared question (`treat_blank_as_no` via config) | reference file |
| `compare_option_labels` | Individual answer option text changed — language-aware (`cfg.language`) | reference file |
| `validate_critical_sets` | Specific named questions must be present with correct Mandatory flag | `critical_sets.yaml` → `exact_sets` |
| `validate_prefix_counts` | Groups must meet minimum counts: CS coping strategies (4/3/3), HDDS (≥3) | `critical_sets.yaml` → `min_count_sets` |
| `validate_crop_harvest` | Crop harvest must be either the minimal set or the full set — no partial | `critical_sets.yaml` → `crop_harvest` |
| `validate_skip_patterns` | Questions referenced in skip patterns must still exist — rules derived from reference, no hardcoding | reference file |


In [8]:
def compare_question_presence(
    current_questions  : pl.DataFrame,
    reference_questions: pl.DataFrame,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """
    Returns (added, removed). Each DataFrame includes `is_optional`
    (True when Q Name starts with 'o_') so the unifier sets severity to
    'info' for optional questions instead of 'medium'/'high'.
    """
    current_q   = current_questions.select("Q Name").unique()
    reference_q = reference_questions.select("Q Name").unique()

    added = (
        current_q.join(reference_q, on="Q Name", how="anti")
        .with_columns(pl.col("Q Name").str.starts_with("o_").alias("is_optional"))
        .sort("Q Name")
    )
    removed = (
        reference_q.join(current_q, on="Q Name", how="anti")
        .with_columns(pl.col("Q Name").str.starts_with("o_").alias("is_optional"))
        .sort("Q Name")
    )
    return added, removed


def compare_mandatory(
    current_questions  : pl.DataFrame,
    reference_questions: pl.DataFrame,
    treat_blank_as_no  : bool = False,
) -> pl.DataFrame:
    """Finds questions where the Mandatory flag differs between current and reference."""
    df = (
        current_questions
        .select(["Q Name", "Mandatory", "excel_row"])
        .join(
            reference_questions.select(["Q Name", "Mandatory", "excel_row"]),
            on="Q Name", how="inner", suffix="_ref",
        )
        .with_columns([
            normalize_mandatory_expr("Mandatory").alias("Mandatory_norm"),
            normalize_mandatory_expr("Mandatory_ref").alias("Mandatory_ref_norm"),
        ])
    )

    if treat_blank_as_no:
        mismatch_filter = (
            (pl.col("Mandatory_norm") != pl.col("Mandatory_ref_norm"))
            & ~((pl.col("Mandatory_norm") == "") & (pl.col("Mandatory_ref_norm") == "no"))
            & ~((pl.col("Mandatory_norm") == "no") & (pl.col("Mandatory_ref_norm") == ""))
        )
    else:
        mismatch_filter = pl.col("Mandatory_norm") != pl.col("Mandatory_ref_norm")

    return (
        df.filter(mismatch_filter)
        .with_columns([
            pl.lit("mandatory_mismatch").alias("issue_type"),
            pl.lit("Mandatory").alias("field"),
        ])
        .sort("Q Name")
    )


def compare_option_labels(
    current_options : pl.DataFrame,
    reference_options: pl.DataFrame,
) -> pl.DataFrame:
    """Joins options on (Q Name, option_code) and flags label differences after normalisation."""
    return (
        current_options
        .join(reference_options, on=["Q Name", "option_code"], how="inner", suffix="_ref")
        .with_columns([
            normalize_text_expr("option_label").alias("option_label_norm"),
            normalize_text_expr("option_label_ref").alias("option_label_ref_norm"),
        ])
        .filter(pl.col("option_label_norm") != pl.col("option_label_ref_norm"))
        .with_columns([
            pl.lit("option_label_mismatch").alias("issue_type"),
            pl.concat_str([pl.lit("option_"), pl.col("option_code").cast(pl.Utf8)]).alias("field"),
        ])
        .sort(["Q Name", "option_code"])
    )


def compare_option_presence(
    current_options : pl.DataFrame,
    reference_options: pl.DataFrame,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """
    Finds options REMOVED from or ADDED TO a question by anti-joining on
    (Q Name, option_code). Detects deleted or inserted answer choices even
    when the question itself is still present in both files.
    Returns (added_options, removed_options).
    """
    key_cols = ["Q Name", "option_code"]

    removed_options = (
        reference_options.select(key_cols + ["option_label"])
        .join(current_options.select(key_cols), on=key_cols, how="anti")
        .with_columns([
            pl.lit("removed_option").alias("issue_type"),
            pl.concat_str([pl.lit("option_"), pl.col("option_code").cast(pl.Utf8)]).alias("field"),
        ])
        .sort(key_cols)
    )

    added_options = (
        current_options.select(key_cols + ["option_label"])
        .join(reference_options.select(key_cols), on=key_cols, how="anti")
        .with_columns([
            pl.lit("added_option").alias("issue_type"),
            pl.concat_str([pl.lit("option_"), pl.col("option_code").cast(pl.Utf8)]).alias("field"),
        ])
        .sort(key_cols)
    )

    return added_options, removed_options


def validate_critical_sets(
    questions_df: pl.DataFrame,
    exact_sets  : dict,
) -> pl.DataFrame:
    """
    Checks that each critical question group is fully present with the expected
    Mandatory value. Rules come from critical_sets.yaml -> exact_sets.

    required=true  -> missing = issue_type 'missing_critical_question', severity HIGH
    required=false -> missing = issue_type 'advisory_question',          severity MEDIUM
    """
    EMPTY_SCHEMA = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }

    if not exact_sets:
        return pl.DataFrame(schema=EMPTY_SCHEMA)

    if "Mandatory_norm" not in questions_df.columns:
        questions_df = questions_df.with_columns(
            normalize_mandatory_expr("Mandatory").alias("Mandatory_norm")
        )

    present_qnames = set(questions_df["Q Name"].to_list())
    issues = []

    for set_name, rules in exact_sets.items():
        required_names   = [r["q_name"] for r in rules if r.get("required", True)]
        present_in_set   = [r["q_name"] for r in rules if r["q_name"] in present_qnames]
        present_required = [q for q in required_names if q in present_qnames]

        if 0 < len(present_required) < len(required_names):
            issues.append({
                "issue_type": "partial_critical_set",
                "set_name"  : set_name, "Q Name": "",
                "field"     : "Q Name",
                "current"   : ", ".join(sorted(present_in_set)),
                "reference" : f"Expected all {len(required_names)} required questions",
                "severity"  : "high", "excel_row": None,
            })

        for rule in rules:
            q_name   = rule["q_name"]
            expected = rule.get("expected_mandatory", "")
            required = rule.get("required", True)

            if q_name not in present_qnames:
                issues.append({
                    "issue_type": "missing_critical_question" if required else "advisory_question",
                    "set_name"  : set_name, "Q Name": q_name,
                    "field"     : "Q Name", "current": "",
                    "reference" : "present",
                    "severity"  : "high" if required else "medium",
                    "excel_row" : None,
                })
                continue

            if not expected:
                continue

            row = (
                questions_df
                .filter(pl.col("Q Name") == q_name)
                .select(["Q Name", "Mandatory", "Mandatory_norm", "excel_row"])
                .to_dicts()
            )
            if not row:
                continue
            row = row[0]
            if row["Mandatory_norm"] != expected:
                issues.append({
                    "issue_type": "critical_mandatory_mismatch",
                    "set_name"  : set_name, "Q Name": q_name,
                    "field"     : "Mandatory",
                    "current"   : row["Mandatory"], "reference": expected,
                    "severity"  : "high", "excel_row": row.get("excel_row"),
                })

    if not issues:
        return pl.DataFrame(schema=EMPTY_SCHEMA)
    return pl.DataFrame(issues)


def validate_prefix_counts(
    current_questions: pl.DataFrame,
    min_count_sets   : dict,
) -> pl.DataFrame:
    """
    Checks that questions with a given prefix appear at least min_count times.
    Covers CS groups (cs_stress_*, cs_crisis_*, cs_emergency_*) and HDDS count.
    Rules come from critical_sets.yaml -> min_count_sets.
    """
    EMPTY_SCHEMA = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if not min_count_sets:
        return pl.DataFrame(schema=EMPTY_SCHEMA)

    all_qnames = current_questions["Q Name"].to_list()
    issues = []
    for set_name, rule in min_count_sets.items():
        prefix    = rule.get("prefix", "")
        min_count = rule.get("min_count", 1)
        desc      = rule.get("description", f"At least {min_count} '{prefix}*' questions required")
        matched   = sorted(q for q in all_qnames if q.startswith(prefix))
        if len(matched) < min_count:
            found_str = f"{len(matched)} found" + (f": {', '.join(matched)}" if matched else " (none)")
            issues.append({
                "issue_type": "below_minimum_count", "set_name": set_name, "Q Name": "",
                "field": "count", "current": found_str, "reference": desc,
                "severity": "high", "excel_row": None,
            })

    if not issues:
        return pl.DataFrame(schema=EMPTY_SCHEMA)
    return pl.DataFrame(issues)


def validate_crop_harvest(
    current_questions: pl.DataFrame,
    crop_rules       : dict,
) -> pl.DataFrame:
    """
    Questionnaire must contain EITHER only the minimal set OR all questions in
    the full set. Rules come from critical_sets.yaml -> crop_harvest.
    """
    EMPTY_SCHEMA = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    if not crop_rules:
        return pl.DataFrame(schema=EMPTY_SCHEMA)

    minimal = set(crop_rules.get("minimal", []))
    full    = set(crop_rules.get("full", []))
    if not minimal and not full:
        return pl.DataFrame(schema=EMPTY_SCHEMA)

    all_qnames     = set(current_questions["Q Name"].to_list())
    relevant_found = {q for q in all_qnames if q in full or q in minimal}

    if relevant_found == minimal or full.issubset(all_qnames):
        return pl.DataFrame(schema=EMPTY_SCHEMA)

    issues = [{
        "issue_type": "crop_harvest_violation", "set_name": "CRP_HARV", "Q Name": "",
        "field": "Q Name",
        "current"  : f"found: {', '.join(sorted(relevant_found)) or 'none'}",
        "reference": f"either only [{', '.join(sorted(minimal))}] OR all of [{', '.join(sorted(full))}]",
        "severity" : "high", "excel_row": None,
    }]
    return pl.DataFrame(issues)


SKIP_PATTERN_COLS = [
    "Default skip patterns & conditional ",   # trailing space is intentional
    "Specify skip pattern variable (from blue text)",
]


def validate_skip_patterns(
    current_questions  : pl.DataFrame,
    reference_questions: pl.DataFrame,
) -> pl.DataFrame:
    """
    Reads skip pattern columns directly from the reference template - no hardcoding.
    For each skip pattern, extracts referenced Q Names and checks they still exist
    in the current questionnaire. Missing = broken skip logic at runtime.
    """
    EMPTY_SCHEMA = {
        "issue_type": pl.Utf8, "set_name": pl.Utf8, "Q Name": pl.Utf8,
        "field": pl.Utf8, "current": pl.Utf8, "reference": pl.Utf8,
        "severity": pl.Utf8, "excel_row": pl.Int64,
    }
    ref_qnames  = set(reference_questions["Q Name"].to_list())
    curr_qnames = set(current_questions["Q Name"].to_list())
    seen, issues = set(), []

    for col in SKIP_PATTERN_COLS:
        if col not in reference_questions.columns:
            continue
        skip_rows = (
            reference_questions
            .filter(pl.col(col).cast(pl.Utf8).fill_null("").str.strip_chars() != "")
            .select(["Q Name", col, "excel_row"])
        )
        for row in skip_rows.iter_rows(named=True):
            source_q  = row["Q Name"]
            skip_text = str(row.get(col) or "")
            excel_row = row.get("excel_row")
            tokens    = re.findall(r'\b[A-Za-z_]\w*\b', skip_text)
            referenced = {t for t in tokens if t in ref_qnames and t != source_q}
            for target_q in referenced:
                key = (source_q, target_q)
                if key in seen:
                    continue
                seen.add(key)
                if target_q not in curr_qnames:
                    issues.append({
                        "issue_type": "broken_skip_pattern", "set_name": "",
                        "Q Name"    : source_q, "field": "skip_pattern",
                        "current"   : f"'{target_q}' missing from current questionnaire",
                        "reference" : skip_text[:120].replace("\n", " "),
                        "severity"  : "high", "excel_row": excel_row,
                    })

    if not issues:
        return pl.DataFrame(schema=EMPTY_SCHEMA)
    return pl.DataFrame(issues)


## Step 8 — Issue unifiers
These functions convert each comparison result into a common schema so they can
be stacked into a single `all_issues` table.

Common columns: `issue_type | set_name | Q Name | field | current | reference | severity | excel_row`

In [9]:
def make_presence_issues(added: pl.DataFrame, removed: pl.DataFrame) -> pl.DataFrame:
    """
    Converts added/removed question lists into the common issues schema.

    Severity rules:
      - Optional questions (o_ prefix): 'info'  — tracked but not a blocker
      - Non-optional added            : 'medium' — unexpected addition, review needed
      - Non-optional removed          : 'high'   — mandatory question missing
    """
    added_issues = added.with_columns([
        pl.lit("added_question").alias("issue_type"),
        pl.lit("").alias("set_name"),
        pl.lit("Q Name").alias("field"),
        pl.lit("present").alias("current"),
        pl.lit("missing_in_reference").alias("reference"),
        pl.lit("info").alias("severity"),
        pl.lit(None).cast(pl.Int64).alias("excel_row"),
    ]).select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])

    removed_issues = removed.with_columns([
        pl.lit("removed_question").alias("issue_type"),
        pl.lit("").alias("set_name"),
        pl.lit("Q Name").alias("field"),
        pl.lit("missing_in_current").alias("current"),
        pl.lit("present").alias("reference"),
        pl.when(pl.col("is_optional")).then(pl.lit("info")).otherwise(pl.lit("high")).alias("severity"),
        pl.lit(None).cast(pl.Int64).alias("excel_row"),
    ]).select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])

    return pl.concat([added_issues, removed_issues], how="vertical")


def make_mandatory_issues(mandatory_diff: pl.DataFrame) -> pl.DataFrame:
    """Converts mandatory mismatches into the common issues schema."""
    return (
        mandatory_diff
        .with_columns([
            pl.lit("").alias("set_name"),
            pl.col("Mandatory").alias("current"),
            pl.col("Mandatory_ref").alias("reference"),
            pl.lit("high").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


def make_option_issues(option_diff: pl.DataFrame) -> pl.DataFrame:
    """Converts option label mismatches into the common issues schema."""
    return (
        option_diff
        .with_columns([
            pl.lit("").alias("set_name"),
            pl.col("option_label").alias("current"),
            pl.col("option_label_ref").alias("reference"),
            pl.lit("medium").alias("severity"),
        ])
        .select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])
    )


# Note: validate_critical_sets, validate_prefix_counts, validate_crop_harvest,
# and validate_skip_patterns already return the common schema directly.


def make_option_presence_issues(added_options: pl.DataFrame, removed_options: pl.DataFrame) -> pl.DataFrame:
    """Converts added/removed options into the common issues schema."""
    added_issues = added_options.with_columns([
        pl.lit("").alias("set_name"),
        pl.col("option_label").alias("current"),
        pl.lit("(not in template)").alias("reference"),
        pl.lit("medium").alias("severity"),
        pl.lit(None).cast(pl.Int64).alias("excel_row"),
    ]).select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])

    removed_issues = removed_options.with_columns([
        pl.lit("").alias("set_name"),
        pl.lit("(removed)").alias("current"),
        pl.col("option_label").alias("reference"),
        pl.lit("high").alias("severity"),
        pl.lit(None).cast(pl.Int64).alias("excel_row"),
    ]).select(["issue_type", "set_name", "Q Name", "field", "current", "reference", "severity", "excel_row"])

    return pl.concat([added_issues, removed_issues], how="vertical")

## Step 9 — Run the full pipeline
Runs all seven checks and assembles a single `all_issues` table sorted by severity (**high → medium → info**).

| Check | Severity |
|---|---|
| Removed non-optional question | high |
| Mandatory flag changed | high |
| Critical set question missing or wrong | high |
| CS count below minimum (4 stress / 3 crisis / 3 emergency) | high |
| Crop harvest rule violated | high |
| Skip pattern references missing question | high |
| Added non-optional question | medium |
| Option label changed | medium |
| Optional (`o_*`) question added or removed | info |

To update which question groups are critical or what the minimum counts are, edit **`critical_sets.yaml`** — no code changes needed.


In [10]:
# ── Run all comparisons ──────────────────────────────────────────────────────────────────────────────
added, removed           = compare_question_presence(current_questions, reference_questions)
mandatory_diff           = compare_mandatory(current_questions, reference_questions,
                                             treat_blank_as_no=cfg.treat_blank_as_no)
option_diff              = compare_option_labels(current_options, reference_options)
added_opts, removed_opts = compare_option_presence(current_options, reference_options)
critical_issues          = validate_critical_sets(current_questions, rules["exact_sets"])
count_issues             = validate_prefix_counts(current_questions, rules["min_count_sets"])
harvest_issues           = validate_crop_harvest(current_questions,  rules["crop_harvest"])
skip_issues              = validate_skip_patterns(current_questions, reference_questions)

# ── Convert to common schema ─────────────────────────────────────────────────────────────────────────────
presence_issues        = make_presence_issues(added, removed)
mandatory_issues       = make_mandatory_issues(mandatory_diff)
option_issues          = make_option_issues(option_diff)
option_presence_issues = make_option_presence_issues(added_opts, removed_opts)

# ── Filter option issues: only report options for questions in BOTH files ──────
_excluded_qnames = set(added["Q Name"].to_list()) | set(removed["Q Name"].to_list())
if _excluded_qnames:
    _excl = list(_excluded_qnames)
    option_issues          = option_issues.filter(~pl.col("Q Name").is_in(_excl))
    option_presence_issues = option_presence_issues.filter(~pl.col("Q Name").is_in(_excl))

# ── Stack all issues ───────────────────────────────────────────────────────────────────────────────────────
all_issues = pl.concat(
    [presence_issues, mandatory_issues, option_issues, option_presence_issues,
     critical_issues, count_issues, harvest_issues, skip_issues],
    how="vertical",
)

# ── Sort: high -> medium -> info ───────────────────────────────────────────────────────────────────────────────
all_issues = (
    all_issues
    .with_columns(
        pl.when(pl.col("severity") == "high"  ).then(pl.lit(0))
        .when(  pl.col("severity") == "medium").then(pl.lit(1))
        .otherwise(pl.lit(2))
        .alias("_sort_order")
    )
    .sort(["_sort_order", "issue_type", "Q Name"])
    .drop("_sort_order")
)

# ── CS downgrade: if count minimum is met, removed CS questions → info ─────────
_passing_prefixes = []
for _set_name, _rule in rules.get("min_count_sets", {}).items():
    if count_issues.filter(pl.col("set_name") == _set_name).height == 0:
        _prefix = _rule.get("prefix", "")
        if _prefix:
            _passing_prefixes.append(_prefix)

if _passing_prefixes:
    all_issues = all_issues.with_columns(
        pl.when(
            (pl.col("issue_type") == "removed_question") &
            pl.col("Q Name").map_elements(
                lambda q: any(q.startswith(p) for p in _passing_prefixes),
                return_dtype=pl.Boolean,
            )
        ).then(pl.lit("info"))
        .otherwise(pl.col("severity"))
        .alias("severity")
    )

# ── Add mandatory_cat column ────────────────────────────────────────────────────────────────────────────────────
# Classifies each question using the Mandatory column and o_ prefix.
# Reference loaded first; current questionnaire values take precedence.
# Categories: mandatory / mandatory-panel / non-mandatory / optional (o_* prefix)
_q_mand: dict = {}
for _src_df in [reference_questions, current_questions]:
    for _row in _src_df.select(["Q Name", "Mandatory"]).to_dicts():
        _q = _row["Q Name"]
        _m = str(_row.get("Mandatory") or "").strip().lower()
        if _q.startswith("o_"):
            _q_mand[_q] = "optional"
        elif "panel" in _m:
            _q_mand[_q] = "mandatory-panel"
        elif _m in ("yes", "y", "true", "1"):
            _q_mand[_q] = "mandatory"
        else:
            _q_mand[_q] = "non-mandatory"

all_issues = all_issues.with_columns(
    pl.col("Q Name").map_elements(
        lambda q: _q_mand.get(q, "") if q else "",
        return_dtype=pl.Utf8,
    ).alias("mandatory_cat")
)

# ── Build found_info for the Critical Sets sheet ─────────────────────────────────────────────
_found_info = {}
for _set_name, _rule in rules.get("min_count_sets", {}).items():
    _prefix  = _rule.get("prefix", "")
    _matched = sorted(q for q in current_questions["Q Name"].to_list() if q.startswith(_prefix))
    _found_info[_set_name] = _matched

# ── Summary ───────────────────────────────────────────────────────────────────────────────────────────────────
print(f"Questions added       : {added.height}")
print(f"Questions removed     : {removed.height}")
print(f"Mandatory mismatches  : {mandatory_diff.height}")
print(f"Option label changes  : {option_diff.height}")
print(f"Options removed       : {removed_opts.height}")
print(f"Options added         : {added_opts.height}")
print(f"Critical set issues   : {critical_issues.height}")
print(f"Count rule violations : {count_issues.height}")
print(f"Crop harvest issues   : {harvest_issues.height}")
print(f"Broken skip patterns  : {skip_issues.height}")
print(f"─────────────────────────────")
print(f"Total issues          : {all_issues.height}")

if all_issues.height > 0:
    print("\nFirst 20 issues:")
    print(all_issues.head(20))
else:
    print("\nNo issues found.")


Questions added       : 30
Questions removed     : 14
Mandatory mismatches  : 4
Option label changes  : 16
Options removed       : 88
Options added         : 333
Critical set issues   : 1
Count rule violations : 0
Crop harvest issues   : 0
Broken skip patterns  : 0
─────────────────────────────
Total issues          : 256

First 20 issues:
shape: (20, 9)
┌────────────┬──────────┬───────────┬───────────┬───┬───────────┬──────────┬───────────┬───────────┐
│ issue_type ┆ set_name ┆ Q Name    ┆ field     ┆ … ┆ reference ┆ severity ┆ excel_row ┆ mandatory │
│ ---        ┆ ---      ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---      ┆ ---       ┆ _cat      │
│ str        ┆ str      ┆ str       ┆ str       ┆   ┆ str       ┆ str      ┆ i64       ┆ ---       │
│            ┆          ┆           ┆           ┆   ┆           ┆          ┆           ┆ str       │
╞════════════╪══════════╪═══════════╪═══════════╪═══╪═══════════╪══════════╪═══════════╪═══════════╡
│ mandatory_ ┆          ┆ Agree     ┆

## Step 10 — Export to Excel

Produces a 4-sheet report:

| Sheet | What it shows |
|---|---|
| **Summary** | Per-set ✅ PASS / ❌ FAIL status at a glance, plus issue counts by category |
| **Critical Sets** | Full detail for all structural issues: CS coping strategies, HDDS, WEALTH, RCSI, crop harvest, broken skip patterns |
| **Template Changes** | Mandatory flag changes, added/removed non-optional questions, option label changes |
| **Optional Questions** | `o_*` questions added or removed (informational only) |

Colour coding: 🔴 red = high severity, 🟠 orange = medium, 🔵 blue = info, 🟢 green = PASS.


In [11]:
from openpyxl import Workbook
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

# ── Colours ──────────────────────────────────────────────────────────────────────────────────────────────
FILL_HIGH    = PatternFill("solid", fgColor="F4CCCC")
FILL_MEDIUM  = PatternFill("solid", fgColor="FCE5CD")
FILL_INFO    = PatternFill("solid", fgColor="CFE2F3")
FILL_PASS    = PatternFill("solid", fgColor="D9EAD3")
FILL_HEADER  = PatternFill("solid", fgColor="274E13")
FILL_SECTION = PatternFill("solid", fgColor="E8F5E9")

FONT_TITLE   = Font(bold=True, size=13, color="274E13")
FONT_HEADER  = Font(bold=True, color="FFFFFF", size=11)
FONT_SECTION = Font(bold=True, color="274E13", size=11)
FONT_NORMAL  = Font(size=10)

THIN_BORDER = Border(
    left=Side(style="thin", color="CCCCCC"),
    right=Side(style="thin", color="CCCCCC"),
    bottom=Side(style="thin", color="CCCCCC"),
)

SEVERITY_FILL = {"high": FILL_HIGH, "medium": FILL_MEDIUM, "info": FILL_INFO}
STATUS_FILL   = {"PASS": FILL_PASS, "FAIL": FILL_HIGH, "WARNING": FILL_MEDIUM}

STRUCTURAL_ISSUE_TYPES = {
    "missing_critical_question", "advisory_question", "partial_critical_set",
    "critical_mandatory_mismatch", "below_minimum_count",
    "crop_harvest_violation", "broken_skip_pattern",
}

# ── Shared helpers ─────────────────────────────────────────────────────────────────────────────────────
def _header_row(ws, row, values):
    for col, val in enumerate(values, 1):
        c = ws.cell(row=row, column=col, value=val)
        c.fill = FILL_HEADER; c.font = FONT_HEADER
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.row_dimensions[row].height = 18

def _section_header(ws, row, title, n_cols):
    ws.merge_cells(start_row=row, start_column=1, end_row=row, end_column=n_cols)
    c = ws.cell(row=row, column=1, value=title)
    c.fill = FILL_SECTION; c.font = FONT_SECTION
    c.alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[row].height = 20

def _data_row(ws, row, n_cols, severity="", fill=None):
    use_fill = fill or SEVERITY_FILL.get(severity)
    for col in range(1, n_cols + 1):
        c = ws.cell(row=row, column=col)
        c.font = FONT_NORMAL; c.border = THIN_BORDER
        c.alignment = Alignment(vertical="top", wrap_text=True)
        if use_fill:
            c.fill = use_fill

def _autofit(ws, mn=12, mx=55):
    for col_cells in ws.columns:
        w = max((len(str(c.value)) if c.value else 0) for c in col_cells)
        ws.column_dimensions[get_column_letter(col_cells[0].column)].width = min(max(w + 2, mn), mx)

def _issues_table(ws, start_row, df):
    """Renders a header + data table. Only columns present in df are shown."""
    COL_MAP = [
        ("issue_type",    "Issue type"),
        ("mandatory_cat", "Type"),
        ("set_name",      "Set"),
        ("Q Name",        "Q Name"),
        ("field",         "Field"),
        ("current",       "Current value"),
        ("reference",     "Reference / rule"),
        ("severity",      "Severity"),
        ("excel_row",     "Excel row"),
    ]
    cols = [(s, d) for s, d in COL_MAP if s in df.columns]
    _header_row(ws, start_row, [d for _, d in cols])
    r = start_row + 1
    if df.height == 0:
        c = ws.cell(row=r, column=1, value="No issues in this category")
        c.font = Font(bold=True, color="274E13", size=10)
        return r + 2
    for row_data in df.to_dicts():
        for col, (src, _) in enumerate(cols, 1):
            ws.cell(row=r, column=col, value=row_data.get(src))
        _data_row(ws, r, len(cols), row_data.get("severity", ""))
        r += 1
    ws.freeze_panes = f"A{start_row + 1}"
    ws.auto_filter.ref = f"A{start_row}:{get_column_letter(len(cols))}{start_row}"
    return r + 1


# ── Per-set status builder ─────────────────────────────────────────────────────────────────────────────
def _set_status(all_issues, rules):
    results = []
    for set_name in rules.get("exact_sets", {}).keys():
        si   = all_issues.filter(pl.col("set_name") == set_name)
        high = si.filter(pl.col("severity") == "high").height
        med  = si.filter(pl.col("severity") == "medium").height
        if high > 0:
            status  = "FAIL"
            details = "; ".join(f"{r['Q Name'] or r['current']}" for r in si.filter(pl.col("severity") == "high").to_dicts())
        elif med > 0:
            status  = "WARNING"
            details = "; ".join(f"{r['Q Name'] or r['current']}" for r in si.to_dicts())
        else:
            status, details = "PASS", "All required questions present and correct"
        results.append({"set": set_name, "status": status, "details": details})
    for set_name, rule in rules.get("min_count_sets", {}).items():
        si = all_issues.filter(pl.col("set_name") == set_name)
        if si.filter(pl.col("severity").is_in(["high", "medium"])).height > 0:
            status, details = "FAIL", si["current"].to_list()[0]
        else:
            status  = "PASS"
            details = f">={rule.get('min_count',0)} {rule.get('prefix','')}* questions confirmed"
        results.append({"set": set_name, "status": status, "details": details})
    if rules.get("crop_harvest"):
        crp = all_issues.filter(pl.col("set_name") == "CRP_HARV")
        if crp.height > 0:
            status, details = "FAIL", crp["current"].to_list()[0]
        else:
            status, details = "PASS", "Crop harvest rule respected"
        results.append({"set": "CRP_HARV", "status": status, "details": details})
    skip = all_issues.filter(pl.col("issue_type") == "broken_skip_pattern")
    results.append({
        "set"    : "SKIP_PATTERNS",
        "status" : "FAIL" if skip.height > 0 else "PASS",
        "details": f"{skip.height} broken reference(s)" if skip.height > 0 else "All skip pattern references intact",
    })
    return results


def _cat_counts(df):
    """Returns counts by mandatory_cat for the four standard categories."""
    cats = ["mandatory", "mandatory-panel", "non-mandatory", "optional"]
    if "mandatory_cat" not in df.columns or df.height == 0:
        return {c: 0 for c in cats}
    return {cat: df.filter(pl.col("mandatory_cat") == cat).height for cat in cats}


# ── Sheet 1: Summary ──────────────────────────────────────────────────────────────────────────────────────────
def write_summary_sheet(wb, all_issues, rules):
    ws = wb.create_sheet("Summary")
    ws.sheet_view.showGridLines = False
    # A=category/set  B=total/status  C=mandatory/details  D=m-panel  E=non-mand.  F=optional  G=severity
    for col, w in zip("ABCDEFG", [32, 8, 13, 10, 13, 10, 14]):
        ws.column_dimensions[col].width = w

    ws.merge_cells("A1:G1")
    ws["A1"] = "Questionnaire Validation Report"
    ws["A1"].font = FONT_TITLE
    ws["A1"].alignment = Alignment(horizontal="left", vertical="center")
    ws.row_dimensions[1].height = 26

    r = 3
    # ── Critical sets ──────────────────────────────────────────────────────────────────────────────────────
    _section_header(ws, r, "CRITICAL SETS STATUS", 7); r += 1
    _header_row(ws, r, ["Critical set", "Status", "Details", "", "", "", ""]); r += 1
    for row_data in _set_status(all_issues, rules):
        fill = STATUS_FILL.get(row_data["status"])
        ws.cell(row=r, column=1, value=row_data["set"])
        ws.cell(row=r, column=2, value=row_data["status"])
        ws.cell(row=r, column=3, value=row_data["details"])
        _data_row(ws, r, 7, fill=fill); r += 1

    r += 1
    # ── Question changes (7 cols with mandatory breakdown) ─────────────────────────────────
    _section_header(ws, r, "QUESTION CHANGES", 7); r += 1
    _header_row(ws, r, ["Category", "Total", "Mandatory", "M-Panel", "Non-mand.", "Optional", "Severity"]); r += 1
    for label, itype, sev_filter, disp_sev in [
        ("Mandatory flag changes",     "mandatory_mismatch", None,   "high"),
        ("Questions removed (high)",   "removed_question",   "high", "high"),
        ("Questions removed (info)",   "removed_question",   "info", "info"),
        ("Questions added",            "added_question",     None,   "info"),
    ]:
        q = all_issues.filter(pl.col("issue_type") == itype)
        if sev_filter:
            q = q.filter(pl.col("severity") == sev_filter)
        counts = _cat_counts(q)
        ws.cell(row=r, column=1, value=label)
        ws.cell(row=r, column=2, value=q.height)
        ws.cell(row=r, column=3, value=counts["mandatory"])
        ws.cell(row=r, column=4, value=counts["mandatory-panel"])
        ws.cell(row=r, column=5, value=counts["non-mandatory"])
        ws.cell(row=r, column=6, value=counts["optional"])
        ws.cell(row=r, column=7, value=disp_sev)
        _data_row(ws, r, 7, disp_sev); r += 1

    r += 1
    # ── Option changes (7 cols with mandatory breakdown) ───────────────────────────────────
    _section_header(ws, r, "OPTION CHANGES (questions present in both files)", 7); r += 1
    _header_row(ws, r, ["Category", "Total", "Mandatory", "M-Panel", "Non-mand.", "Optional", "Severity"]); r += 1
    for label, itype, disp_sev in [
        ("Options removed",       "removed_option",        "high"),
        ("Options added",         "added_option",          "medium"),
        ("Option labels changed", "option_label_mismatch", "medium"),
    ]:
        q = all_issues.filter(pl.col("issue_type") == itype)
        counts = _cat_counts(q)
        ws.cell(row=r, column=1, value=label)
        ws.cell(row=r, column=2, value=q.height)
        ws.cell(row=r, column=3, value=counts["mandatory"])
        ws.cell(row=r, column=4, value=counts["mandatory-panel"])
        ws.cell(row=r, column=5, value=counts["non-mandatory"])
        ws.cell(row=r, column=6, value=counts["optional"])
        ws.cell(row=r, column=7, value=disp_sev)
        _data_row(ws, r, 7, disp_sev); r += 1


# ── Sheet 2: Critical Sets ─────────────────────────────────────────────────────────────────────────────────────
def write_critical_sets_sheet(wb, all_issues, found_info=None):
    ws = wb.create_sheet("Critical Sets")
    ws.sheet_view.showGridLines = False

    df = all_issues.filter(
        pl.col("issue_type").is_in(list(STRUCTURAL_ISSUE_TYPES))
    ).sort(["set_name", "severity", "Q Name"])

    _section_header(ws, 1, "CRITICAL SETS — Structural validation issues", 8)
    next_row = _issues_table(ws, 2, df)

    if found_info:
        next_row += 1
        _section_header(ws, next_row, "QUESTIONS FOUND IN CURRENT QUESTIONNAIRE", 3)
        next_row += 1
        _header_row(ws, next_row, ["Set", "Count found", "Questions"])
        next_row += 1
        for set_name, questions in found_info.items():
            ws.cell(row=next_row, column=1, value=set_name)
            ws.cell(row=next_row, column=2, value=len(questions))
            ws.cell(row=next_row, column=3, value=", ".join(questions) if questions else "none found")
            _data_row(ws, next_row, 3, "info" if questions else "high")
            next_row += 1

    _autofit(ws)


# ── Sheet 3: Question Changes ───────────────────────────────────────────────────────────────────────────────────
def write_question_changes_sheet(wb, all_issues):
    """
    All question-level changes sorted high→info. The 'Type' column shows
    mandatory / mandatory-panel / non-mandatory / optional for each question.
    The 'Field' column is omitted — it is implicit from the issue type.
    """
    ws = wb.create_sheet("Question Changes")
    ws.sheet_view.showGridLines = False

    df = all_issues.filter(
        pl.col("issue_type").is_in([
            "mandatory_mismatch", "added_question", "removed_question",
        ])
    )
    df = (
        df.with_columns(
            pl.when(pl.col("severity") == "high"  ).then(pl.lit(0))
            .when(  pl.col("severity") == "medium").then(pl.lit(1))
            .otherwise(pl.lit(2))
            .alias("_s")
        )
        .sort(["_s", "issue_type", "Q Name"])
        .drop("_s")
    )
    for col in ("set_name", "field"):
        if col in df.columns:
            df = df.drop(col)

    _section_header(ws, 1, "QUESTION CHANGES — Presence and mandatory flag (all severities)", 8)
    _issues_table(ws, 2, df)
    _autofit(ws)


# ── Sheet 4: Option Changes ────────────────────────────────────────────────────────────────────────────────────
def write_option_changes_sheet(wb, all_issues):
    """
    Option-level changes for questions present in both files. The 'Field'
    column shows option_1 / option_2 / … so you know which answer choice is
    affected. The 'Type' column shows the mandatory category of the parent question.
    """
    ws = wb.create_sheet("Option Changes")
    ws.sheet_view.showGridLines = False

    df = all_issues.filter(
        pl.col("issue_type").is_in([
            "option_label_mismatch", "added_option", "removed_option",
        ])
    )
    df = (
        df.with_columns(
            pl.when(pl.col("severity") == "high"  ).then(pl.lit(0))
            .when(  pl.col("severity") == "medium").then(pl.lit(1))
            .otherwise(pl.lit(2))
            .alias("_s")
        )
        .sort(["_s", "Q Name", "field"])
        .drop("_s")
    )
    if "set_name" in df.columns:
        df = df.drop("set_name")

    _section_header(ws, 1, "OPTION CHANGES — Answer options for questions in both files", 8)
    _issues_table(ws, 2, df)
    _autofit(ws)


# ── Main export ────────────────────────────────────────────────────────────────────────────────────────────────
def export_validation_report(all_issues, result_file, rules, found_info=None):
    wb = Workbook()
    wb.remove(wb.active)
    write_summary_sheet(wb, all_issues, rules)
    write_critical_sets_sheet(wb, all_issues, found_info=found_info)
    write_question_changes_sheet(wb, all_issues)
    write_option_changes_sheet(wb, all_issues)
    wb.save(result_file)
    print(f"Report saved to: {result_file}")
    print(f"Sheets: {wb.sheetnames}")


export_validation_report(
    all_issues  = all_issues,
    result_file = run["result_file"],
    rules       = rules,
    found_info  = _found_info,
)


Report saved to: C:\Questionnaire_Validation\output\GeoPoll_FAO_FR_DRC_R11_test_geopoll_validated.xlsx
Sheets: ['Summary', 'Critical Sets', 'Question Changes', 'Option Changes']
